# Phase 7: Robustness Evaluation (Simulated Missing Modalities)
Validates trained checkpoints under partial input dropout scenarios.

In [ ]:
print("[BACKGROUND] Loading test split features...")
import os
import sys
import torch
import pandas as pd
from torch.utils.data import DataLoader

sys.path.append(os.path.abspath('../'))
from src.models.fleximodal_moe import FlexiModalMoE
from src.training.trainer import MultimodalTrainer
print("[STATUS] Validation imports successful.")

In [ ]:
print("[BACKGROUND] Reconstructing datasets from test split subjects...")
# Reload data directly from processed test partition
df_test = pd.read_csv('../data/processed/test/test_synced.csv')

# Redefine TestDataset inline
class TestDataset(torch.utils.data.Dataset):
    def __init__(self, df, seq_len=5):
        self.df = df.reset_index(drop=True)
        self.seq_len = seq_len
        # Locate feature columns dynamically supporting both real and mock columns
        face_cands = ['left_ear', 'right_ear', 'avg_ear', 'blink_velocity', 'brow_descent_left', 'brow_descent_right', 'brow_asymmetry', 'lip_compression', 'jaw_tension', 'mouth_corner_pull', 'forehead_tension', 'face_height_norm', 'head_tilt', 'temporal_x_var', 'temporal_y_var', 'eye_openness_ratio', 'landmark_confidence', 'nose_wrinkle']
        self.face_cols = [c for c in face_cands if c in df.columns]
        if len(self.face_cols) == 0:
            self.face_cols = [c for c in df.columns if c.startswith('face_') and c not in ['face_mask']]
            
        voice_cands = ['f0_mean', 'f0_std', 'f0_range', 'jitter_percent', 'shimmer_db', 'hnr', 'speaking_rate_proxy', 'voice_intensity', 'high_freq_ratio', 'spectral_flux', 'pause_ratio', 'voiced_fraction']
        self.voice_cols = [c for c in voice_cands if c in df.columns]
        if len(self.voice_cols) == 0:
            self.voice_cols = [c for c in df.columns if c.startswith('voice_') and c not in ['voice_mask']]
            
        physio_cands = ['ecg_rate_mean', 'ecg_hrv_rmssd', 'ecg_hrv_sdnn', 'eda_scl_mean', 'resp_rate_mean']
        self.physio_cols = [c for c in physio_cands if c in df.columns]
        if len(self.physio_cols) == 0:
            self.physio_cols = [c for c in df.columns if c.startswith('physio_') and c not in ['physio_mask']]
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        # Force conversion to float32 and handle NaNs dynamically from left join
        face_nans = self.df.loc[idx, self.face_cols].isna().any()
        voice_nans = self.df.loc[idx, self.voice_cols].isna().any()
        physio_nans = self.df.loc[idx, self.physio_cols].isna().any()
        
        face_mask = torch.tensor(0.0 if face_nans else 1.0, dtype=torch.float32)
        voice_mask = torch.tensor(0.0 if voice_nans else 1.0, dtype=torch.float32)
        physio_mask = torch.tensor(0.0 if physio_nans else 1.0, dtype=torch.float32)
        
        face_vals = np.nan_to_num(self.df.loc[idx, self.face_cols].values.astype(np.float32), nan=0.0)
        voice_vals = np.nan_to_num(self.df.loc[idx, self.voice_cols].values.astype(np.float32), nan=0.0)
        physio_vals = np.nan_to_num(self.df.loc[idx, self.physio_cols].values.astype(np.float32), nan=0.0)
        
        face_val = torch.tensor(face_vals, dtype=torch.float32).unsqueeze(0).repeat(self.seq_len, 1)
        voice_val = torch.tensor(voice_vals, dtype=torch.float32).unsqueeze(0).repeat(self.seq_len, 1)
        physio_val = torch.tensor(physio_vals, dtype=torch.float32).unsqueeze(0).repeat(self.seq_len, 1)
        label = torch.tensor(int(self.df.loc[idx, 'label']), dtype=torch.long)
        
        return {
            'face': face_val,
            'voice': voice_val,
            'physio': physio_val,
            'face_mask': face_mask,
            'voice_mask': voice_mask,
            'physio_mask': physio_mask,
            'label': label
        }

test_ds = TestDataset(df_test)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)
print(f"[INFO] Test loader configured with {len(test_ds)} sample windows.")

In [ ]:
print("[BACKGROUND] Commencing missing-modality robustness ablation audits for all 5 models...")
import os
import time
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from src.models.baselines import EarlyFusionClassifier, GatedFusionClassifier, CrossAttentionFusionClassifier
from src.models.fleximodal_moe import FlexiModalMoE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
face_dim = len(test_ds.face_cols)
voice_dim = len(test_ds.voice_cols)
physio_dim = len(test_ds.physio_cols)

models_info = [
    {"name": "Early Fusion Baseline",        "class": "EarlyFusionClassifier",        "ckpt": "../outputs/checkpoints/best_early.pt"},
    {"name": "Gated Fusion Baseline",        "class": "GatedFusionClassifier",        "ckpt": "../outputs/checkpoints/best_gated.pt"},
    {"name": "Cross-Attention Fusion",       "class": "CrossAttentionFusionClassifier", "ckpt": "../outputs/checkpoints/best_attention.pt"},
    {"name": "Standard MoE Fusion",          "class": "FlexiModalMoE",                "ckpt": "../outputs/checkpoints/best_moe_standard.pt"},
    {"name": "Robust FlexiModal MoE",        "class": "FlexiModalMoE",                "ckpt": "../outputs/checkpoints/best_moe_robust.pt"}
]

scenarios = [
    {"name": "All Modalities Present", "face": 1.0, "voice": 1.0, "physio": 1.0},
    {"name": "Missing Face/Video",     "face": 0.0, "voice": 1.0, "physio": 1.0},
    {"name": "Missing Voice/Audio",    "face": 1.0, "voice": 0.0, "physio": 1.0},
    {"name": "Missing Physio Logs",    "face": 1.0, "voice": 1.0, "physio": 0.0},
]

results = []
for m_info in models_info:
    print(f"\n[AUDIT] Evaluating: {m_info['name']}...")
    if not os.path.exists(m_info["ckpt"]):
        print(f"  [WARNING] Checkpoint {m_info['ckpt']} not found. Skipping.")
        continue
        
    # Instantiate model
    if m_info["class"] == "EarlyFusionClassifier":
        model = EarlyFusionClassifier(face_dim, voice_dim, physio_dim)
    elif m_info["class"] == "GatedFusionClassifier":
        model = GatedFusionClassifier(face_dim, voice_dim, physio_dim)
    elif m_info["class"] == "CrossAttentionFusionClassifier":
        model = CrossAttentionFusionClassifier(face_dim, voice_dim, physio_dim)
    elif m_info["class"] == "FlexiModalMoE":
        model = FlexiModalMoE(face_dim, voice_dim, physio_dim, num_experts=3)
        
    # Load weights
    ckpt = torch.load(m_info["ckpt"], map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()
    
    # Trainable parameters count
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Measure latency
    latencies = []
    dummy_face = torch.randn(1, 5, face_dim).to(device)
    dummy_voice = torch.randn(1, 5, voice_dim).to(device)
    dummy_physio = torch.randn(1, 5, physio_dim).to(device)
    dummy_mask = torch.tensor([1.0]).to(device)
    
    # Warm-up
    for _ in range(5):
        with torch.no_grad():
            if m_info["class"] == "FlexiModalMoE":
                _ = model(dummy_face, dummy_voice, dummy_physio, dummy_mask, dummy_mask, dummy_mask)
            else:
                _ = model(dummy_face, dummy_voice, dummy_physio)
                
    # Measure forward pass time
    for _ in range(50):
        start_time = time.time()
        with torch.no_grad():
            if m_info["class"] == "FlexiModalMoE":
                _ = model(dummy_face, dummy_voice, dummy_physio, dummy_mask, dummy_mask, dummy_mask)
            else:
                _ = model(dummy_face, dummy_voice, dummy_physio)
        latencies.append((time.time() - start_time) * 1000.0)
    avg_latency = np.mean(latencies)
    
    # Evaluate on test dataset splits per missing modality scenario
    for scen in scenarios:
        correct = 0
        total = 0
        for batch in test_loader:
            face_x = batch['face'].to(device)
            voice_x = batch['voice'].to(device)
            physio_x = batch['physio'].to(device)
            labels = batch['label'].to(device)
            
            f_mask = torch.tensor([scen['face']]*face_x.size(0), device=device)
            v_mask = torch.tensor([scen['voice']]*voice_x.size(0), device=device)
            p_mask = torch.tensor([scen['physio']]*physio_x.size(0), device=device)
            
            with torch.no_grad():
                if m_info["class"] == "FlexiModalMoE":
                    logits, _ = model(face_x, voice_x, physio_x, f_mask, v_mask, p_mask)
                else:
                    # Manual masking for baselines
                    face_in = face_x * f_mask.view(-1, 1, 1)
                    voice_in = voice_x * v_mask.view(-1, 1, 1)
                    physio_in = physio_x * p_mask.view(-1, 1, 1)
                    logits = model(face_in, voice_in, physio_in)
                preds = torch.argmax(logits, dim=-1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        all_labels = np.array(all_labels)
        all_preds = np.array(all_preds)
        
        acc = accuracy_score(all_labels, all_preds)
        macro_f1 = f1_score(all_labels, all_preds, average='macro')
        
        # Per-class precision, recall, f1-score
        prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, labels=[0, 1], zero_division=0)
        
        results.append({
            "Model": m_info["name"],
            "Parameters": num_params,
            "Latency (ms)": f"{avg_latency:.3f}ms",
            "Scenario": scen["name"],
            "Accuracy": f"{acc*100:.2f}%",
            "Macro F1": f"{macro_f1*100:.2f}%",
            "Relaxed Prec (C0)": f"{prec[0]*100:.2f}%",
            "Relaxed Rec (C0)": f"{rec[0]*100:.2f}%",
            "Relaxed F1 (C0)": f"{f1[0]*100:.2f}%",
            "Stressed Prec (C1)": f"{prec[1]*100:.2f}%",
            "Stressed Rec (C1)": f"{rec[1]*100:.2f}%",
            "Stressed F1 (C1)": f"{f1[1]*100:.2f}%"
        })
        
results_df = pd.DataFrame(results)
os.makedirs('../reports/evaluation/', exist_ok=True)
results_df.to_csv('../reports/evaluation/robustness_metrics.csv', index=False)
print("[STATUS] All 5 models evaluated and ablation study metrics saved to robustness_metrics.csv.")